# 2VA — Análise Estatística: Ensemble vs Modelo Único

Lê os dois CSVs gerados pelos notebooks anteriores e responde:  
**O ensemble (H1) é estatisticamente superior ao modelo único (H0)?**

- Teste: Wilcoxon signed-rank (dados pareados, distribuição desconhecida)  
- Métricas: F1 macro (sentimento) e MAE (rating)  
- Nível de significância: α = 0.05

## Passo 1 — Setup + carregar resultados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from pathlib import Path

RESULTS_DIR = Path("../results")
ALPHA = 0.05   # nível de significância

single   = pd.read_csv(RESULTS_DIR / "single_model_results.csv")
ensemble = pd.read_csv(RESULTS_DIR / "ensemble_results.csv")

# alinha pela seed — garante que os pares são corretos para o Wilcoxon
single   = single.sort_values("seed").reset_index(drop=True)
ensemble = ensemble.sort_values("seed").reset_index(drop=True)

assert (single["seed"] == ensemble["seed"]).all(), "Seeds desalinhadas — verifique os CSVs"

print(f"Simulacoes carregadas: {len(single)} pares (seed 0..{single['seed'].max()})")
print(f"\nC_alt (modelo unico):\n{single[['f1_macro','mae']].describe().round(4)}")
print(f"\nC (ensemble):\n{ensemble[['f1_macro','mae']].describe().round(4)}")

## Passo 2 — Teste de Wilcoxon

Teste não-paramétrico para amostras pareadas. Escolhido porque:
- Não assume distribuição normal dos resultados
- Os 30 pares são naturalmente pareados pela mesma seed (mesmo split de dados)
- `alternative='greater'` testa se ensemble > modelo único (H1 direcional)

In [ ]:
def wilcoxon_report(name, a, b, alternative, higher_is_better=True):
    """
    Testa se `a` é estatisticamente diferente de `b`.
    alternative='greater' → H1: a > b
    alternative='less'    → H1: a < b  (para MAE, menor é melhor)
    """
    stat, p = stats.wilcoxon(a, b, alternative=alternative)
    diff_mean = (a - b).mean()
    rejeita   = p < ALPHA

    simbolo = ">" if alternative == "greater" else "<"
    direcao = "ensemble SUPERIOR" if rejeita else "sem diferenca significativa"

    print(f"{'='*52}")
    print(f"  {name}")
    print(f"  ensemble  : {a.mean():.4f} ± {a.std():.4f}")
    print(f"  modelo    : {b.mean():.4f} ± {b.std():.4f}")
    print(f"  diff media: {diff_mean:+.4f}  (ensemble {simbolo} modelo)")
    print(f"  W={stat:.1f}  p={p:.4f}  α={ALPHA}")
    print(f"  -> {'REJEITA H0' if rejeita else 'NAO rejeita H0'} — {direcao}")
    return {"metrica": name, "p_value": p, "rejeita_h0": rejeita, "diff_mean": diff_mean}

results_stat = []

# F1 — maior é melhor → H1: ensemble_f1 > single_f1
r1 = wilcoxon_report(
    "F1 macro (sentimento)",
    ensemble["f1_macro"], single["f1_macro"],
    alternative="greater",
)
results_stat.append(r1)

# MAE — menor é melhor → H1: ensemble_mae < single_mae
r2 = wilcoxon_report(
    "MAE (rating)",
    ensemble["mae"], single["mae"],
    alternative="less",
)
results_stat.append(r2)

## Passo 3 — Visualizações

Boxplot comparativo + histograma de diferenças pareadas — gráficos para o relatório IEEE.

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

COLORS = {"single": "#5B9BD5", "ensemble": "#ED7D31"}
LABELS = {"single": "Modelo único (C_alt)", "ensemble": "Ensemble (C)"}

# ── Boxplot F1 ────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.boxplot(
    [single["f1_macro"], ensemble["f1_macro"]],
    tick_labels=[LABELS["single"], LABELS["ensemble"]],
    patch_artist=True,
    boxprops=dict(facecolor="none"),
    medianprops=dict(color="black", linewidth=2),
)
ax1.set_title("F1 macro — Sentimento", fontweight="bold")
ax1.set_ylabel("F1 macro")
ax1.grid(axis="y", alpha=0.3)

# ── Boxplot MAE ───────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.boxplot(
    [single["mae"], ensemble["mae"]],
    tick_labels=[LABELS["single"], LABELS["ensemble"]],
    patch_artist=True,
    boxprops=dict(facecolor="none"),
    medianprops=dict(color="black", linewidth=2),
)
ax2.set_title("MAE — Rating", fontweight="bold")
ax2.set_ylabel("MAE")
ax2.grid(axis="y", alpha=0.3)

# ── Histograma diferenças F1 ──────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
diff_f1 = ensemble["f1_macro"] - single["f1_macro"]
ax3.hist(diff_f1, bins=10, color=COLORS["ensemble"], edgecolor="white", alpha=0.8)
ax3.axvline(0, color="black", linestyle="--", linewidth=1.2, label="sem diferença")
ax3.axvline(diff_f1.mean(), color="red", linestyle="-", linewidth=1.5, label=f"média={diff_f1.mean():+.4f}")
ax3.set_title("Diferença F1 (ensemble − modelo único)", fontweight="bold")
ax3.set_xlabel("Δ F1 macro")
ax3.legend(fontsize=8)
ax3.grid(axis="y", alpha=0.3)

# ── Histograma diferenças MAE ─────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
diff_mae = ensemble["mae"] - single["mae"]
ax4.hist(diff_mae, bins=10, color=COLORS["single"], edgecolor="white", alpha=0.8)
ax4.axvline(0, color="black", linestyle="--", linewidth=1.2, label="sem diferença")
ax4.axvline(diff_mae.mean(), color="red", linestyle="-", linewidth=1.5, label=f"média={diff_mae.mean():+.4f}")
ax4.set_title("Diferença MAE (ensemble − modelo único)", fontweight="bold")
ax4.set_xlabel("Δ MAE")
ax4.legend(fontsize=8)
ax4.grid(axis="y", alpha=0.3)

fig.suptitle("Ensemble vs Modelo Único — 30 simulações", fontsize=14, fontweight="bold", y=1.01)
plt.savefig(RESULTS_DIR / "comparativo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva em results/comparativo.png")

## Passo 4 — Conclusão

In [ ]:
print("=" * 60)
print("  RESULTADOS — ENSEMBLE vs MODELO ÚNICO (30 seeds)")
print("=" * 60)
print(f"  {'Métrica':<28} {'Modelo Único':>14} {'Ensemble':>12} {'Δ médio':>10}")
print(f"  {'-'*28} {'-'*14} {'-'*12} {'-'*10}")

diff_f1  = ensemble["f1_macro"] - single["f1_macro"]
diff_mae = ensemble["mae"]      - single["mae"]

print(f"  {'F1 macro':<28} {single['f1_macro'].mean():.4f}±{single['f1_macro'].std():.4f}  {ensemble['f1_macro'].mean():.4f}±{ensemble['f1_macro'].std():.4f}  {diff_f1.mean():+.4f}")
print(f"  {'MAE (rating)':<28} {single['mae'].mean():.4f}±{single['mae'].std():.4f}  {ensemble['mae'].mean():.4f}±{ensemble['mae'].std():.4f}  {diff_mae.mean():+.4f}")

print()
print(f"  {'Métrica':<28} {'W':>8} {'p-value':>12} {'Decisão':>20}")
print(f"  {'-'*28} {'-'*8} {'-'*12} {'-'*20}")

for r in results_stat:
    decisao = "REJEITA H0 (H1)" if r["rejeita_h0"] else "nao rejeita H0"
    print(f"  {r['metrica']:<28} {'-':>8} {r['p_value']:>12.6f}  {decisao:>20}")

print()
votos_h1 = sum(r["rejeita_h0"] for r in results_stat)

if votos_h1 == 2:
    print("  VEREDICTO FINAL: H1 ACEITA em ambas as metricas.")
    print("  Ensemble (A+B+C) é estatisticamente SUPERIOR ao modelo unico (A+B+C_alt).")
    print(f"  Ganho no F1: {diff_f1.mean():+.4f} ({diff_f1.mean()/single['f1_macro'].mean()*100:+.1f}%)")
    print(f"  Reducao MAE: {diff_mae.mean():+.4f} ({diff_mae.mean()/single['mae'].mean()*100:+.1f}%)")
elif votos_h1 == 1:
    print("  VEREDICTO FINAL: H1 PARCIAL — ensemble superior em 1 de 2 metricas.")
else:
    print("  VEREDICTO FINAL: H0 MANTIDA — ensemble NAO demonstrou superioridade.")

print("=" * 60)

---

## Resultados Finais

### Comparativo de Performance (30 seeds)

| Métrica | Modelo Único (C_alt) | Ensemble (C) | Δ médio |
|---------|---------------------|--------------|---------|
| F1 macro | 0.6353 ± 0.0491 | 0.6932 ± 0.0563 | +0.0579 (+9.1%) |
| MAE | 0.1568 ± 0.0161 | 0.1425 ± 0.0118 | −0.0142 (−9.1%) |

### Wilcoxon Signed-Rank Test (α = 0.05)

| Métrica | W | p-value | Decisão |
|---------|---|---------|---------|
| F1 macro (H1: ensemble > single) | 424.0 | **0.000009** | REJEITA H0 |
| MAE (H1: ensemble < single) | 53.5 | **0.000116** | REJEITA H0 |

### Veredicto

> **H1 ACEITA** — O ensemble (A+B+C) é estatisticamente superior ao modelo único (A+B+C_alt) em ambas as métricas, com p << 0.05.  
> A média de 3 modelos independentes reduz a variância e cancela erros idiossincráticos de cada treino individual.